<a href="https://colab.research.google.com/github/itsdev-ai/Transformer-Neural-Network/blob/main/Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

**Device Configuration**

---

This function checks if a GPU (CUDA) is available. If yes, the model will train on GPU, otherwise it falls back to CPU. This is a standard practice for PyTorch training.

In [2]:
def get_device():
  return torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

**Scaled Dot-Product**

---
This is the core attention mechanism from the paper "Attention Is All You Need".
Formula: Attention(Q,K,V) = softmax(Q_K^T / sqrt(d_k)) _ V
Masking is used to ignore padding tokens and to prevent the decoder from looking at future tokens.


In [3]:
def scale_dot_product(q,k,v,mask=None):
  d_k=q.size()[-1]
  scaled=torch.matmul(q,k.transpose(-1,-2))/math.sqrt(d_k)
  if mask is not None:
    scaled=scaled.permute(1,0,2,3)+mask
    scaled=scaled.permute(1,0,2,3)
  attention=F.softmax(scaled,dim=-1)
  value=torch.matmul(attention,v)
  return value,attention

**Positional Encoding**

---
Since Transformer has no recurrence, we need to inject information about the position of tokens. We use sine and cosine functions of different frequencies as proposed in the paper. This gives the model a sense of word order.


In [4]:
class PositionalEncoding(nn.Module):
  def __init__(self,d_model,max_sequence_length):
    super().__init__()
    self.d_model=d_model
    self.max_sequence_length=max_sequence_length
  def forward(self):
    even_i=torch.arange(0,self.d_model,2).float()
    denomination=torch.pow(10000,even_i/self.d_model)
    positional=(torch.arange(self.max_sequence_length).reshape(self.max_sequence_length,1))
    even_PE=torch.sin(positional/denomination)
    odd_PE=torch.cos(positional/denomination)
    stacked=torch.stack([even_PE,odd_PE],dim=2)
    PE=torch.flatten(stacked,start_dim=1,end_dim=2)
    return PE

**Embedding**

---

This layer combines two things: 1. Token Embedding converts word IDs into dense vectors. 2. Positional Encoding is added to it. Dropout is used for regularization.

In [5]:
class Embedding(nn.Module):
  def __init__(self,vocab_size,d_model,max_sequence_lenght,dropout=0.1):
    super().__init__()
    self.vocab_size=vocab_size
    self.d_model=d_model
    self.max_sequence_lenght=max_sequence_lenght
    self.dropout=nn.Dropout(p=dropout)
    self.embedding=nn.Embedding(vocab_size,d_model)
    self.positional_encoder=PositionalEncoding(d_model,max_sequence_lenght)

  def forward(self,x):
    x=self.embedding(x)
    pos=self.positional_encoder().to(get_device())
    result=self.dropout(x+pos)
    return result

**MultiheadAttention**


---
Instead of one attention, we split Q, K, V into multiple heads. This allows the model to focus on different parts of the sentence at the same time. We create Q,K,V using a single linear layer for efficiency and then reshape.


In [6]:
class MultiheadAttention(nn.Module):
  def __init__(self,d_model,num_head):
    super().__init__()
    self.d_model=d_model
    self.num_head=num_head
    self.head_dim=d_model//num_head
    self.qkv_layer=nn.Linear(d_model,3*d_model)
    self.linear_layer=nn.Linear(d_model,d_model)

  def forward(self,x,mask):
    batch_size,max_sequence_length,d_model=x.size()
    qkv=self.qkv_layer(x)
    qkv=qkv.reshape(batch_size,max_sequence_length,self.num_head,3*self.head_dim)
    qkv=qkv.permute((0,2,1,3))
    q,k,v=qkv.chunk(3,dim=-1)
    value,attention=scale_dot_product(q,k,v,mask)
    values=value.permute(0,2,1,3).reshape(batch_size,max_sequence_length,self.num_head*self.head_dim)
    out=self.linear_layer(values)
    return out

**Layer-Normalization**

In [7]:
class LayerNormalization(nn.Module):
  def __init__(self,parameter_shape,eps=1e-5):
    super().__init__()
    self.parameter_shape=parameter_shape
    self.eps=eps
    self.gamma=nn.Parameter(torch.ones(parameter_shape))
    self.beta=nn.Parameter(torch.zeros(parameter_shape))

  def forward(self,input):
    dims=[-(i+1) for i in range(len(self.parameter_shape))]
    mean=input.mean(dim=dims,keepdim=True)
    var=((input-mean)**2).mean(dim=dims,keepdim=True)
    std=(var+self.eps).sqrt()
    y=(input-mean)/std
    out=self.gamma*y+self.beta
    return out

**PositionWise Feed Forward Network**

---
**Whai is this? :**
This is the 2nd sub-layer inside both Encoder and Decoder. After Attention looks at other words, this network processes each position independently.


**Formula:**
 max(0, x_W1 + b1)_W2 + b2


In [8]:
class PositionwiseFeedForward(nn.Module):
  def __init__(self,d_model,hidden,drop_prob=0.1):
    super(PositionwiseFeedForward,self).__init__()
    self.linear1=nn.Linear(d_model,hidden)
    self.linear2=nn.Linear(hidden,d_model)
    self.dropout=nn.Dropout(p=drop_prob)
    self.relu=nn.ReLU()
  def forward(self,x):
    x=self.linear1(x)
    x=self.relu(x)
    x=self.dropout(x)
    x=self.linear2(x)
    return x

**Encoder Layer**

---
One Encoder Layer consists of: Multi-Head Attention -> Add & Norm (Residual Connection + LayerNorm) -> Feed Forward Network -> Add & Norm. The residual connection residual_x = x.clone() helps in avoiding vanishing gradients.


In [9]:
class EncoderLayer(nn.Module):
  def __init__(self,d_model,hidden,num_head,drop_prob):
    super(EncoderLayer,self).__init__()
    self.attention=MultiheadAttention(d_model=d_model,num_head=num_head)
    self.norm1=LayerNormalization(parameter_shape=[d_model])
    self.dropout1=nn.Dropout(p=drop_prob)
    self.ffn=PositionwiseFeedForward(d_model=d_model,hidden=hidden,drop_prob=drop_prob)
    self.norm2=LayerNormalization(parameter_shape=[d_model])
    self.dropout2=nn.Dropout(p=drop_prob)

  def forward(self,x,self_attention_mask):
    residual_x=x.clone()
    x=self.attention(x,mask=self_attention_mask)
    x=self.dropout1(x)
    x=residual_x+x
    x=self.norm1(x)

    residual_x=x.clone()
    x=self.ffn(x)
    x=self.dropout2(x)
    x=residual_x+x
    x=self.norm2(x)
    return x

**Sequential Encoder**


---
SequentialEncoder stacks N number of EncoderLayer blocks. Encoder first converts input tokens to embeddings and then passes them through all encoder layers.


In [10]:
class SequentialEncoder(nn.Sequential):
  def forward(self,*input):
    x,self_attention_mask=input

    for module in self._modules.values():
      x=module(x,self_attention_mask)
    return x

**Encoder**

In [11]:
class Encoder(nn.Module):
  def __init__(self,
               d_model,
               hidden,
               num_head,
               drop_prob,
               num_layers,
               max_sequence_length,
               input_vocab_size):
    super().__init__()
    self.embedding=Embedding(input_vocab_size,d_model,max_sequence_length)
    self.layer=SequentialEncoder(*[EncoderLayer(d_model,hidden,num_head,drop_prob) for _ in range(num_layers)])

  def forward(self,x,self_attention_mask):
    x=self.embedding(x)
    x=self.layer(x,self_attention_mask)
    return x

**Multi-head Cross Attention**

---
This is used in the Decoder. Here Query comes from the Decoder (target sentence), while Key and Value come from the Encoder output (source sentence). This is how the decoder attends to the encoder's information.


In [12]:
class MultiheadCrossAttention(nn.Module):
  def __init__(self,d_model,num_head):
    super().__init__()
    self.d_model=d_model
    self.num_head=num_head
    self.head_dim=d_model//num_head
    self.kv_layer=nn.Linear(d_model,2*d_model)
    self.q_layer=nn.Linear(d_model,d_model)
    self.linear_layer=nn.Linear(d_model,d_model)

  def forward(self,x,y,mask=None):
    batch_size,max_sequence_lenght,d_model=x.size()
    kv=self.kv_layer(x)
    q=self.q_layer(y)
    kv=kv.reshape(batch_size,max_sequence_lenght,self.num_head,2*self.head_dim)
    q=q.reshape(batch_size,max_sequence_lenght,self.num_head,self.head_dim)
    kv=kv.permute(0,2,1,3)
    q=q.permute(0,2,1,3)
    k,v=kv.chunk(2,dim=-1)
    values,attention=scale_dot_product(q,k,v,mask)
    values=values.permute(0,2,1,3).reshape(batch_size,max_sequence_lenght,d_model)
    out=self.linear_layer(values)
    return out


**Decoder Layer**

---
Decoder Layer has 3 sub-layers: 1. Masked Multi-Head Self-Attention 2. Multi-Head Cross-Attention 3. Position-wise Feed Forward. Each sub-layer has a residual connection and LayerNorm.


In [13]:
class DecoderLayer(nn.Module):
  def __init__(self,d_model,hidden,num_head,drop_prob):
    super(DecoderLayer,self).__init__()
    self.maskedmultiheadattention=MultiheadAttention(d_model=d_model,num_head=num_head)
    self.dropout1=nn.Dropout(p=drop_prob)
    self.norm1=LayerNormalization(parameter_shape=[d_model])

    self.multiheadcrossattention=MultiheadCrossAttention(d_model=d_model,num_head=num_head)
    self.dropout2=nn.Dropout(p=drop_prob)
    self.norm2=LayerNormalization(parameter_shape=[d_model])

    self.ffd=PositionwiseFeedForward(d_model=d_model,hidden=hidden,drop_prob=drop_prob)
    self.dropout3=nn.Dropout(p=drop_prob)
    self.norm3=LayerNormalization(parameter_shape=[d_model])

  def forward(self,x,y,self_attention_mask,cross_attention_mask):
    residual_y=y.clone()
    y=self.maskedmultiheadattention(y,mask=self_attention_mask)
    y=self.dropout1(y)
    y=residual_y+y
    y=self.norm1(y)

    residual_y=y.clone()
    y=self.multiheadcrossattention(x,y,mask=cross_attention_mask)
    y=self.dropout2(y)
    y=residual_y+y
    y=self.norm2(y)

    residual_y=y.clone()
    y=self.ffd(y)
    y=self.dropout3(y)
    y=residual_y+y
    y=self.norm3(y)
    return y

In [14]:
class SequentialDecoder(nn.Sequential):
  def forward(self,*input):
    x,y,self_attention_mask,cross_attention_mask=input
    for module in self._modules.values():
      y=module(x,y,self_attention_mask,cross_attention_mask)
    return y

**Decoder**

In [15]:
class Decoder(nn.Module):
  def __init__(self,
               d_model,
               hidden,
               num_head,
               drop_prob,
               num_layers,
               max_sequence_length,
               target_vocab_size):
    super().__init__()
    self.embedding=Embedding(target_vocab_size,d_model,max_sequence_length)
    self.layer=SequentialDecoder(*[DecoderLayer(d_model,hidden,num_head,drop_prob) for _ in range(num_layers)])

  def forward(self,x,y,self_attention_mask,cross_attention_mask):
    y=self.embedding(y)
    y=self.layer(x,y,self_attention_mask,cross_attention_mask)
    return y

**Transformer**

---
This is the complete Transformer architecture. It combines Encoder, Decoder, and a final Linear layer to project decoder output to vocabulary size. The forward method takes source (x), target (y), and masks as input.


In [16]:
class Transformer(nn.Module):
  def __init__(self,
               d_model,
               num_head,
               hidden,
               drop_prob,
               num_layers,
               max_sequence_length,
               input_vocab_size,
               target_vocab_size,
               vocab_size_for_linear):
    super().__init__()
    self.encoder=Encoder(d_model,hidden,num_head,drop_prob,num_layers,max_sequence_length,input_vocab_size)
    self.decoder=Decoder(d_model,hidden,num_head,drop_prob,num_layers,max_sequence_length,target_vocab_size)
    self.linear=nn.Linear(d_model,vocab_size_for_linear)
    self.Device=torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

  def forward (self,x,y,encoder_self_attention_mask=None,decoder_self_attention_mask=None,decoder_cross_attention_mask=None):
    x=self.encoder(x,encoder_self_attention_mask)
    out=self.decoder(x,y,decoder_self_attention_mask,decoder_cross_attention_mask)
    out=self.linear(out)
    return out